# 02 — All Four Methods: European Call Greeks

Four estimators for Greek computation:

| Method | Key idea | Requires f to be... |
|--------|----------|---------------------|
| **Malliavin** | Weight π from IBP formula | Measurable (any payoff) |
| **Finite difference** | Re-simulate with perturbed S₀ | Nothing, but variance explodes as h→0 |
| **Pathwise / IPA** | Differentiate f(S_T) directly | Differentiable |
| **Likelihood ratio** | Score function of transition density | Measurable |

For smooth payoffs, all four converge.  For discontinuous payoffs, only Malliavin and
LR have finite variance.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import time

from mgreeks.models.gbm import GeometricBrownianMotion
from mgreeks.payoffs.european import EuropeanCall
from mgreeks.simulation import MonteCarloEngine
from mgreeks.greeks import (
    MalliavinGreeks, FiniteDifferenceGreeks, PathwiseGreeks, LikelihoodRatioGreeks,
)
from mgreeks.greeks.analytical import bs_delta, bs_gamma, bs_vega

S0, K, T = 100.0, 100.0, 1.0
r, q, sigma = 0.05, 0.02, 0.20
n_paths, seed = 50_000, 42

model = GeometricBrownianMotion(r=r, q=q, sigma=sigma)
payoff = EuropeanCall(K)
engine = MonteCarloEngine(model, n_paths=n_paths, n_steps=1, rng_seed=seed)

mall = MalliavinGreeks(model, engine)
fd   = FiniteDifferenceGreeks(model, engine, bump_size=0.01, bump_type="relative")
pw   = PathwiseGreeks(model, engine)
lr   = LikelihoodRatioGreeks(model, engine)

bs = {
    "delta": bs_delta(S0, K, T, r, q, sigma, "call"),
    "gamma": bs_gamma(S0, K, T, r, q, sigma),
    "vega":  bs_vega(S0, K, T, r, q, sigma),
}
print(f"BS analytics: delta={bs['delta']:.4f}  gamma={bs['gamma']:.4f}  vega={bs['vega']:.4f}")


In [ ]:
## Side-by-side comparison

results = {}
for greek in ["delta", "gamma", "vega"]:
    results[greek] = {}
    for name, estimator in [("Malliavin", mall), ("FD", fd), ("Pathwise", pw), ("LR", lr)]:
        try:
            t0 = time.perf_counter()
            res = getattr(estimator, greek)(payoff, S0, T)
            elapsed = time.perf_counter() - t0
            results[greek][name] = {"value": res["value"], "se": res["std_error"], "t": elapsed}
        except (NotImplementedError, Exception):
            results[greek][name] = None

print(f"{'Greek':<8} {'Method':<12} {'Estimate':>10} {'SE':>10} {'Bias':>10} {'Time(s)':>8}")
print("-" * 65)
for greek in ["delta", "gamma", "vega"]:
    truth = bs[greek]
    for name, res in results[greek].items():
        if res is None:
            print(f"{greek:<8} {name:<12} {'N/A':>10}")
            continue
        bias = res['value'] - truth
        print(f"{greek:<8} {name:<12} {res['value']:>10.5f} {res['se']:>10.5f} {bias:>10.5f} {res['t']:>8.3f}")
    print()


In [ ]:
## Convergence: SE vs n_paths  (delta, European call)

n_grid = [1_000, 3_000, 10_000, 30_000, 100_000]
se_mall, se_fd = [], []

for n in n_grid:
    eng_n = MonteCarloEngine(model, n_paths=n, n_steps=1, rng_seed=seed)
    m = MalliavinGreeks(model, eng_n)
    f_est = FiniteDifferenceGreeks(model, eng_n, bump_size=0.01, bump_type="relative")
    se_mall.append(m.delta(payoff, S0, T)["std_error"])
    se_fd.append(f_est.delta(payoff, S0, T)["std_error"])

n_arr = np.array(n_grid, dtype=float)
ref = se_mall[0] * np.sqrt(n_grid[0]) / np.sqrt(n_arr)

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(n_arr, se_mall, "b-o", label="Malliavin SE")
ax.loglog(n_arr, se_fd,   "r--s", label="FD SE (h=1%)")
ax.loglog(n_arr, ref, "k:", lw=1, label="1/√n reference")
ax.set_xlabel("n_paths"); ax.set_ylabel("Std Error")
ax.set_title("Convergence: European Call Delta"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("02_convergence.png", dpi=100, bbox_inches="tight")
plt.show()
print("Both methods converge at 1/√n.  FD has lower SE (FD wins for smooth payoffs with CRN).")


In [ ]:
## FD bump-size dilemma: European call vs Digital call

from mgreeks.payoffs.european import DigitalCall
from mgreeks.greeks.analytical import bs_digital_delta

digital = DigitalCall(K)
h_grid  = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1]

se_euro_fd, se_dig_fd = [], []
for h in h_grid:
    fd_h = FiniteDifferenceGreeks(model, engine, bump_size=h, bump_type="relative")
    se_euro_fd.append(fd_h.delta(payoff,  S0, T)["std_error"])
    se_dig_fd.append( fd_h.delta(digital, S0, T)["std_error"])

# Malliavin SE (constant — no h)
se_mall_euro = mall.delta(payoff,  S0, T)["std_error"]
se_mall_dig  = mall.delta(digital, S0, T)["std_error"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("FD Bump-Size Dilemma: SE vs h", fontsize=13)

for ax, se_fd_arr, se_mall_ref, title in [
    (axes[0], se_euro_fd, se_mall_euro, "European Call Delta"),
    (axes[1], se_dig_fd,  se_mall_dig,  "Digital Call Delta"),
]:
    ax.loglog(h_grid, se_fd_arr, "r-o", label="FD SE")
    ax.axhline(se_mall_ref, color="b", lw=2, label=f"Malliavin SE = {se_mall_ref:.5f}")
    ax.set_xlabel("Bump size h"); ax.set_ylabel("Std Error")
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("02_bump_dilemma.png", dpi=100, bbox_inches="tight")
plt.show()
print("Digital: FD SE diverges as h→0.  Malliavin SE is constant and finite.")
